# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The queue is ML-07's real, already-produced `work/outputs/baseline_action_score.csv`** —
this notebook doesn't invent a new ranking, it packages the one that already exists into
something a human can act on. Every row carries exactly one reason code
(`CTR_GAP_VS_POSITION`) and one action (`REWRITE_TITLE_META`): *"this page ranks well but
converts far below what pages at its position typically do — the fix to try first is the
title/meta snippet, not a content rewrite or a technical audit."* If ML-08's model ends up
outperforming the baseline (per ML-09's real numbers), swap the source file here to the
model's ranked output — the packaging in this notebook doesn't change either way.


In [1]:
import pandas as pd
queue = pd.read_csv("work/outputs/baseline_action_score.csv")
print("Queue rows:", len(queue))
print(queue[["client_hash_id","content_hash_id","action_score","reason_code","action"]].head(10))


Queue rows: 45854
            client_hash_id           content_hash_id  action_score  \
0  client_e547b89c05043229  content_545bb6cc7081ded3  22794.316036   
1  client_e547b89c05043229  content_eadb33b5df496f4a  22289.337294   
2  client_8ddc46da5414ffd8  content_943dc881428182b8  12502.744690   
3  client_e547b89c05043229  content_9ef3d7516483e665  11123.208973   
4  client_8ddc46da5414ffd8  content_cca099da6c658785   6308.779357   
5  client_e547b89c05043229  content_0e03de7680314cd5   4901.482381   
6  client_e547b89c05043229  content_61215c724c8220ae   4769.849879   
7  client_73cda7b4e4f265ea  content_e9856d7d976aa034   4608.121198   
8  client_e547b89c05043229  content_8d7d99f109e19aa2   4109.631430   
9  client_8ddc46da5414ffd8  content_7471467133493ce6   3287.857231   

           reason_code              action  
0  CTR_GAP_VS_POSITION  REWRITE_TITLE_META  
1  CTR_GAP_VS_POSITION  REWRITE_TITLE_META  
2  CTR_GAP_VS_POSITION  REWRITE_TITLE_META  
3  CTR_GAP_VS_POSITION  REWRITE

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** an SEO/content team lead, to decide which pages a writer/editor looks at
this week — a triage list, not an auto-pilot.

**Where it stops being valid:**
- Pages with a SERP feature (featured snippet, PAA box) eating the click — a title rewrite
  won't fix a click that never had a chance to land on this result (flagged concretely in
  ML-07's top-20 review, e.g. the row with search_volume 4,400 and near-zero CTR).
- Branded/navigational queries, where low CTR is normal, not a fixable problem.
- Small clients with thin history — ML-08's error analysis (once run) should say whether the
  model, if used, is measurably worse for low-data clients; the baseline doesn't have this
  problem since it doesn't need historical training data, but it does structurally favor
  high-traffic clients in the ranking itself (already flagged in ML-07's weak-picks section).
- Anything older than the current month's data pull — this is a snapshot, not a live feed.


In [2]:
print("Failure conditions: SERP features, branded queries, stale search_volume, thin-history clients.")


Failure conditions: SERP features, branded queries, stale search_volume, thin-history clients.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a human checks:** does this query actually have a competing SERP
feature right now (quick manual search)? Is the query branded/navigational? Does the page's
current title/meta already look reasonable, or is there an obvious dated/broken snippet?

**Never automate:** publishing a rewritten title/meta without a human reading it first — the
score identifies *where* to look, not *what to write*. Also never automate bulk-editing pages
belonging to a client without checking whether that client has out-of-band context (a
seasonal campaign, a recent migration) the data can't see.


In [3]:
print("No-go: auto-publishing rewrites. Always human-reviewed before anything goes live.")


No-go: auto-publishing rewrites. Always human-reviewed before anything goes live.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signals that this queue has gone stale:**
- The position-bucket CTR curve (`expected_ctr_by_bucket`, computed fresh each month) shifts
  meaningfully month over month — if it does, last month's benchmark is the wrong yardstick for
  this month's pages.
- A rewritten page's actual CTR doesn't move after being acted on — if a sample of "fixed"
  pages shows no lift over a few months, the rule's core assumption (title/meta rewrite closes
  the CTR gap) needs re-testing, not just re-running.
- If ML-08's model is ever put into production instead of the baseline: retrain when
  Precision@50 on fresh held-out clients drops meaningfully below what ML-09 measured at
  launch — that's the signal the feature relationships have shifted, not just noise.


In [4]:
print("Retrain/refresh trigger: re-check expected_ctr_by_bucket monthly; watch for drift.")


Retrain/refresh trigger: re-check expected_ctr_by_bucket monthly; watch for drift.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Writes the artifacts the capstone paper embeds directly** — the top-20 table (already
reviewed row-by-row in ML-07) and a chart of the CTR-by-position-bucket curve that the whole
rule is built on, since that's the single most load-bearing piece of evidence in this project.


In [6]:
import os
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 1. Save top-20 recommendations
# ============================================================

os.makedirs("work/outputs", exist_ok=True)

top20 = (
    queue
    .sort_values("action_score", ascending=False)
    .head(20)
    .copy()
)

top20.to_csv(
    "work/outputs/top20_for_paper.csv",
    index=False
)


# ============================================================
# 2. Load content performance data
# ============================================================

# This is the path confirmed to work in your ML-06/ML-07 setup.
DATA_DIR = "../../"

perf_path = f"{DATA_DIR}/fact_content_daily_performance_sample.parquet"

perf = pd.read_parquet(
    perf_path,
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

print(f"Loaded performance data: {len(perf):,} rows")


# ============================================================
# 3. Aggregate performance by client-content pair
# ============================================================

agg = (
    perf
    .groupby(
        ["client_hash_id", "content_hash_id"]
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_pos=("gsc_avg_position", "mean")
    )
    .reset_index()
)


# ============================================================
# 4. Keep sufficiently observed content
# ============================================================

agg = agg[agg["impressions"] >= 50].copy()

# Avoid division-by-zero
agg["ctr"] = agg["clicks"] / agg["impressions"]

print(f"Aggregated content pairs: {len(agg):,}")


# ============================================================
# 5. Create position buckets
# ============================================================

bins = [0, 3, 5, 10, 20, 50, 1000]

labels = [
    "1-3",
    "3-5",
    "5-10",
    "10-20",
    "20-50",
    "50+"
]

agg["pos_bucket"] = pd.cut(
    agg["avg_pos"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


# ============================================================
# 6. Calculate weighted CTR for each position bucket
# ============================================================

bucket_ctr = (
    agg
    .groupby("pos_bucket", observed=True)
    .apply(
        lambda g: g["clicks"].sum() / g["impressions"].sum()
    )
    .reindex(labels)
)

print("\nCTR by position bucket:")
print(bucket_ctr)


# ============================================================
# 7. Create the CTR-position plot
# ============================================================

fig, ax = plt.subplots(figsize=(6, 4))

bucket_ctr.plot(
    kind="bar",
    ax=ax
)

ax.set_xlabel("Average position bucket")
ax.set_ylabel("CTR")
ax.set_title(
    "CTR by position bucket "
    "(the curve the baseline is built on)"
)

plt.tight_layout()

plt.savefig(
    "work/outputs/ctr_by_position_bucket.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close(fig)


# ============================================================
# 8. Final confirmation
# ============================================================

print("\nWrote:")
print("  work/outputs/top20_for_paper.csv")
print("  work/outputs/ctr_by_position_bucket.png")

Loaded performance data: 11,694,072 rows
Aggregated content pairs: 120,681

CTR by position bucket:
pos_bucket
1-3      0.044120
3-5      0.006791
5-10     0.003530
10-20    0.004102
20-50    0.002632
50+      0.000644
dtype: float64

Wrote:
  work/outputs/top20_for_paper.csv
  work/outputs/ctr_by_position_bucket.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.